In [3]:
import mlflow
from mlflow.tracking import MlflowClient

In [7]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = MlflowClient()

# Trouver le meilleur Run selon le F1
# (baseline = meilleur compromis budget moyen)
experiment = client.get_experiment_by_name("customer-churn-platform")

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1 DESC"],
    max_results=1
)

meilleur_run = runs[0]
run_id = meilleur_run.info.run_id
f1 = meilleur_run.data.metrics["f1"]
approche = meilleur_run.data.params["approche"]

print(f"Meilleur Run : {meilleur_run.info.run_name}")
print(f"Approche     : {approche}")
print(f"F1 Score     : {f1:.4f}")

# Enregistrer dans le Registry
model_uri  = f"runs:/{run_id}/model_baseline"
model_name = "customer-churn-classifier"

registered = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

# Ajouter description
client.update_registered_model(
    name=model_name,
    description="CatBoost Baseline : meilleur compromis Recall/Precision pour budget moyen"
)

# Staging puis Production
client.set_registered_model_alias(
    name=model_name, alias="staging", version=registered.version
)
print("Modèle en STAGING")

client.set_registered_model_alias(
    name=model_name, alias="production", version=registered.version
)
print("Modèle en PRODUCTION")

Successfully registered model 'customer-churn-classifier'.
2026/08/17 03:44:48 WARNING mlflow.tracking._model_registry.fluent: Run with id b4c41e9705074af4a83a533402d5c218 has no artifacts at artifact path 'model_baseline', registering model based on models:/m-151f028afb15448f9ce4e756725ba408 instead


Meilleur Run : CatBoost_Baseline
Approche     : baseline
F1 Score     : 0.6102
Modèle en STAGING
Modèle en PRODUCTION


Created version '1' of model 'customer-churn-classifier'.


In [ ]:
FROM python:3.12-slim

WORKDIR /app

COPY requirements-api.txt .
RUN pip install -r requirements-api.txt

COPY main.py .
COPY notebooks/mlflow.db ./notebooks/
COPY notebooks/mlruns ./notebooks/mlruns

EXPOSE 7860

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]